<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/Fisc_Dom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!find /content/drive/MyDrive -type f -name "theta_timeseries.csv" 2>/dev/null | head -n 50

/content/drive/MyDrive/CGTAD2/_colab_checkpoint_2026-02-11/DTT_EXP/runs/run_0001/theta_timeseries.csv
/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
/content/drive/MyDrive/DTT_EXP/runs/run_0001/theta_timeseries.csv
/content/drive/MyDrive/Colab_Backup/20260212_work/DTT_EXP/runs/run_0001/theta_timeseries.csv


In [ ]:
# =========================
# EWS (Event-level Early Warning) — Colab one-cell runnable
# Re-generate Fig.9-like PDFs and Table 7-like LaTeX from theta_timeseries.csv
# Definition-consistent with Sec. 6.3:
#   V_t := IG(theta_t) / (eps + STI_proxy(t))
#   labels y_t = 1 on lead windows [t_e - L, t_e), else 0
#   threshold calibrated to hit TARGET_FPR on pre-event baseline negatives
# Outputs:
#   outputs_ews/FigEWS_<Episode>.pdf
#   outputs_ews/table_ews_eval.tex
# =========================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# (OPTIONAL) Mount Google Drive (uncomment if needed)
# -------------------------
# from google.colab import drive
# drive.mount('/content/drive')

# -------------------------
# CONFIG (EDIT HERE)
# -------------------------
OUTDIR = Path("outputs_ews")
OUTDIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-6
TARGET_FPR = 0.05
ROLL_WIN = 12   # monthly -> 12; if your theta series is high-frequency, increase (e.g., 60~250)

# Lead window length (calendar days). Will convert into "steps" using inferred sampling interval.
LEAD_DAYS_DEFAULT = 365  # 12 months

# --- Episodes configuration ---
# You can supply either:
#  (A) direct CSV paths, OR
#  (B) a ROOT folder to auto-find "theta_timeseries.csv" under it.
#
# Expected CSV columns (minimum): theta1, theta2
# Optional columns: date/time (any), IG (if you already computed IG series)
#
# Event times: provide as date strings if your CSV has dates, otherwise they will be mapped to nearest index.
EPISODES = [
    dict(
        name="Takahashi",
        csv_path="PATH/TO/Takahashi/theta_timeseries.csv",  # <-- EDIT
        date_col=None,   # set e.g. "date" if exists (auto-detect if None)
        # Events per Sec 6.3: Feb 1936 (2.26 incident)
        event_dates=["1936-02-26"],
        lead_days=LEAD_DAYS_DEFAULT,

        # STI normal (Table 1: nfis = (1,0)^T) => use theta1 increment
        normal="theta1",

        # IG mode (Appendix D.1: wedge upper-left rectangle)
        ig_mode="upper_left_wedge",
        ig_params=dict(C_fiscal=None, C_army=None,  # if None -> inferred by quantiles
                       q_C_fiscal=0.20, q_C_army=0.80),
    ),
    dict(
        name="Abenomics",
        csv_path="PATH/TO/Abenomics/theta_timeseries.csv",  # <-- EDIT
        date_col=None,
        # Events per Sec 6.3: YCC adoption (2016) and FX breakout (2022)
        event_dates=["2016-09-21", "2022-10-01"],  # <-- adjust to your chosen markers
        lead_days=LEAD_DAYS_DEFAULT,

        # STI normal (Table 1: YCC pinning in yield dimension) => use theta2 increment
        normal="theta2",

        # IG mode for holonomic wedge (Appendix D.2): line θ2≈1 (pinning trap).
        # Pure line gives IG=0 always, so we use a "pinning band" thickness δ to quantify trap intensity.
        ig_mode="pinning_band",
        ig_params=dict(target=1.0, band=0.02),  # <-- tune band (0.01~0.05)
    ),
    dict(
        name="ScenarioS",
        csv_path="PATH/TO/ScenarioS/theta_timeseries.csv",  # <-- EDIT
        date_col=None,
        # Projected event path: choose an endpoint marker (or a specific designed "failure" time)
        event_dates=["2099-12-31"],  # placeholder if no dates; will map to last index
        lead_days=LEAD_DAYS_DEFAULT,

        normal="theta2",

        # In the draft, Scenario S often uses a large IG (e.g., 0.4 in Appendix C.3 narrative).
        # If you have IG column in CSV, set ig_mode="from_csv" and ensure column IG exists.
        ig_mode="constant",
        ig_params=dict(value=0.4),
    ),
]

# Optional: if you want auto-find instead of manually setting csv_path above:
AUTO_FIND = False
SEARCH_ROOT = "/content/drive/MyDrive"  # set to your run folder for faster search

# -------------------------
# Utilities
# -------------------------
def _autodetect_date_col(df: pd.DataFrame):
    for c in df.columns:
        lc = c.lower()
        if lc in ["date", "datetime", "time", "timestamp", "t"]:
            return c
    return None

def _safe_datetime_index(df: pd.DataFrame, date_col: str | None, start: str, end: str):
    if date_col is None:
        date_col = _autodetect_date_col(df)
    if date_col is not None and date_col in df.columns:
        idx = pd.to_datetime(df[date_col], errors="coerce")
        if idx.notna().sum() >= max(5, int(0.5 * len(df))):
            df = df.copy()
            df.index = idx
            df = df.sort_index()
            return df, True
    # fallback: synthetic DateTimeIndex across [start,end]
    df = df.copy()
    df.index = pd.date_range(start=start, end=end, periods=len(df))
    return df, False

def _infer_dt_days(index: pd.DatetimeIndex) -> float:
    if not isinstance(index, pd.DatetimeIndex) or len(index) < 3:
        return 1.0
    diffs = np.diff(index.view("int64"))  # ns
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return 1.0
    med_ns = np.median(diffs)
    return float(med_ns / (1e9 * 3600 * 24))  # days

def _nearest_index(dt_index: pd.DatetimeIndex, date_str: str):
    # If placeholder far-future date, map to last
    try:
        d = pd.to_datetime(date_str)
    except Exception:
        return len(dt_index) - 1
    if d > dt_index.max():
        return len(dt_index) - 1
    if d < dt_index.min():
        return 0
    pos = int(np.argmin(np.abs((dt_index - d).values.astype("timedelta64[ns]").astype(np.int64))))
    return pos

def _rolling_var(x: np.ndarray, win: int):
    s = pd.Series(x)
    v = s.rolling(win, min_periods=max(3, win//3)).var()
    return v.to_numpy()

def _sti_proxy(df: pd.DataFrame, normal: str, roll_win: int):
    th1 = df["theta1"].to_numpy(dtype=float)
    th2 = df["theta2"].to_numpy(dtype=float)
    d1 = np.diff(th1, prepend=th1[0])
    d2 = np.diff(th2, prepend=th2[0])

    if normal == "theta1":
        dn = d1
    elif normal == "theta2":
        dn = d2
    else:
        # custom vector "n=(n1,n2)" string not supported; keep simple
        dn = d1

    sti = _rolling_var(dn, roll_win)
    # numerical safety: non-negative and fill NaNs
    sti = np.where(np.isfinite(sti), sti, np.nan)
    if np.all(np.isnan(sti)):
        sti = np.zeros_like(dn)
    else:
        # forward-fill then zeros
        sti = pd.Series(sti).fillna(method="ffill").fillna(0.0).to_numpy()
    return sti

def _ig_from_upper_left_wedge(df: pd.DataFrame, params: dict):
    th1 = df["theta1"].to_numpy(dtype=float)
    th2 = df["theta2"].to_numpy(dtype=float)

    C_fiscal = params.get("C_fiscal", None)
    C_army   = params.get("C_army", None)
    q1 = params.get("q_C_fiscal", 0.20)
    q2 = params.get("q_C_army", 0.80)

    if C_fiscal is None:
        C_fiscal = float(np.quantile(th1, q1))
    if C_army is None:
        C_army = float(np.quantile(th2, q2))

    inside = (th1 <= C_fiscal) & (th2 >= C_army)
    ig = np.zeros_like(th1, dtype=float)
    # distance to complement of wedge (Appendix D.1 rectangle): min(C_fiscal - th1, th2 - C_army)
    ig_inside = np.minimum(C_fiscal - th1[inside], th2[inside] - C_army)
    ig[inside] = np.maximum(ig_inside, 0.0)

    meta = dict(C_fiscal=C_fiscal, C_army=C_army, q_C_fiscal=q1, q_C_army=q2)
    return ig, meta

def _ig_from_pinning_band(df: pd.DataFrame, params: dict):
    th2 = df["theta2"].to_numpy(dtype=float)
    target = float(params.get("target", 1.0))
    band   = float(params.get("band", 0.02))
    # "trap intensity": larger when closer to target within band
    # IG := max(0, band - |theta2-target|)
    ig = np.maximum(0.0, band - np.abs(th2 - target))
    meta = dict(target=target, band=band)
    return ig, meta

def _ig_from_constant(df: pd.DataFrame, params: dict):
    v = float(params.get("value", 0.0))
    ig = np.full(len(df), v, dtype=float)
    return ig, dict(value=v)

def _compute_ig(df: pd.DataFrame, ig_mode: str, ig_params: dict):
    if ig_mode == "from_csv":
        if "IG" not in df.columns:
            raise ValueError("ig_mode='from_csv' but column 'IG' not found.")
        ig = df["IG"].to_numpy(dtype=float)
        return ig, dict(source="csv:IG")
    if ig_mode == "upper_left_wedge":
        return _ig_from_upper_left_wedge(df, ig_params)
    if ig_mode == "pinning_band":
        return _ig_from_pinning_band(df, ig_params)
    if ig_mode == "constant":
        return _ig_from_constant(df, ig_params)
    raise ValueError(f"Unknown ig_mode={ig_mode}")

def _build_labels(index: pd.DatetimeIndex, event_dates: list[str], lead_days: float):
    dt_days = _infer_dt_days(index)
    lead_steps = max(1, int(round(lead_days / max(dt_days, 1e-6))))
    y = np.zeros(len(index), dtype=int)
    event_idxs = []
    for ed in event_dates:
        ei = _nearest_index(index, ed)
        event_idxs.append(ei)
        s = max(0, ei - lead_steps)
        e = max(0, ei)  # exclude event time itself
        if e > s:
            y[s:e] = 1
    return y, event_idxs, lead_steps, dt_days

def _calibrate_threshold(scores: np.ndarray, y: np.ndarray, event_idxs: list[int], lead_steps: int, target_fpr: float):
    # baseline negatives: pre-first-event region excluding lead windows
    valid = np.isfinite(scores)
    if len(event_idxs) == 0:
        neg_mask = (y == 0) & valid
    else:
        first_e = min(event_idxs)
        baseline_end = max(0, first_e - lead_steps)
        neg_mask = (np.arange(len(y)) < baseline_end) & (y == 0) & valid
        if neg_mask.sum() < 50:
            # fallback: all negatives
            neg_mask = (y == 0) & valid
    neg_scores = scores[neg_mask]
    if len(neg_scores) < 10:
        return np.nan, int(neg_mask.sum())
    tau = float(np.quantile(neg_scores, 1.0 - target_fpr))
    return tau, int(neg_mask.sum())

def _eval_metrics(scores: np.ndarray, y: np.ndarray):
    valid = np.isfinite(scores) & np.isfinite(y)
    yv = y[valid].astype(int)
    sv = scores[valid].astype(float)
    if len(np.unique(yv)) < 2:
        return np.nan, np.nan, int((yv==1).sum()), int((yv==0).sum())
    roc = float(roc_auc_score(yv, sv))
    pr  = float(average_precision_score(yv, sv))
    return roc, pr, int((yv==1).sum()), int((yv==0).sum())

def _lead_at_threshold(scores: np.ndarray, index: pd.DatetimeIndex, event_idxs: list[int], lead_steps: int, tau: float, dt_days: float):
    # compute lead for each event: first crossing within lead window
    leads_months = []
    for ei in event_idxs:
        s = max(0, ei - lead_steps)
        e = max(0, ei)
        window_scores = scores[s:e]
        if not np.isfinite(tau) or len(window_scores) == 0:
            leads_months.append(np.nan)
            continue
        cross = np.where(window_scores > tau)[0]
        if len(cross) == 0:
            leads_months.append(np.nan)
            continue
        t_cross = s + int(cross[0])
        lead_steps_i = ei - t_cross
        lead_days_i = lead_steps_i * dt_days
        leads_months.append(float(lead_days_i / 30.4375))
    if len(leads_months) == 0:
        return np.nan, []
    return float(np.nanmean(leads_months)), leads_months

def _plot_episode(name: str, df: pd.DataFrame, ig: np.ndarray, sti: np.ndarray, V: np.ndarray,
                  event_idxs: list[int], lead_steps: int, tau: float):
    t = df.index
    fig = plt.figure(figsize=(10, 7))

    ax1 = plt.subplot(3,1,1)
    ax1.plot(t, ig, linewidth=1.2)
    ax1.set_ylabel("IG(θt)")
    ax1.grid(True, alpha=0.25)

    ax2 = plt.subplot(3,1,2, sharex=ax1)
    ax2.plot(t, sti, linewidth=1.2)
    ax2.set_ylabel("STI_proxy(t)")
    ax2.grid(True, alpha=0.25)

    ax3 = plt.subplot(3,1,3, sharex=ax1)
    ax3.plot(t, V, linewidth=1.2)
    if np.isfinite(tau):
        ax3.axhline(tau, linestyle="--", linewidth=1.0)
    ax3.set_ylabel("V_t")
    ax3.set_xlabel("Time")
    ax3.grid(True, alpha=0.25)

    # markers + lead windows
    for ei in event_idxs:
        te = t[ei]
        for ax in (ax1, ax2, ax3):
            ax.axvline(te, linewidth=1.0)
            ts = t[max(0, ei - lead_steps)]
            ax.axvspan(ts, te, alpha=0.12)

    plt.suptitle(f"EWS: {name}", y=0.98)
    plt.tight_layout()

    out = OUTDIR / f"FigEWS_{name}.pdf"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    return out

def _sanitize(name: str):
    return "".join([c for c in name if c.isalnum() or c in ("_", "-")])

def _autofind_csv(root: str, pattern: str = "theta_timeseries.csv", max_hits: int = 20):
    hits = []
    rootp = Path(root)
    if not rootp.exists():
        return hits
    for p in rootp.rglob(pattern):
        hits.append(p)
        if len(hits) >= max_hits:
            break
    # sort by mtime desc
    hits.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return hits

# -------------------------
# RUN
# -------------------------
rows = []
generated_figs = []

for ep in EPISODES:
    name = _sanitize(ep["name"])
    csv_path = ep["csv_path"]

    if AUTO_FIND or (csv_path is None) or (not Path(csv_path).exists()):
        hits = _autofind_csv(SEARCH_ROOT, "theta_timeseries.csv", max_hits=50)
        if len(hits) == 0:
            raise FileNotFoundError(f"[{name}] CSV not found. Set ep['csv_path'] or disable AUTO_FIND.")
        # heuristic: pick the newest hit; you can refine by filtering with episode name in the path
        chosen = None
        for h in hits:
            if ep["name"].lower() in str(h).lower():
                chosen = h
                break
        if chosen is None:
            chosen = hits[0]
        csv_path = str(chosen)
        print(f"[AUTO] {name}: using {csv_path}")

    df = pd.read_csv(csv_path)
    # enforce theta columns
    if "theta1" not in df.columns or "theta2" not in df.columns:
        raise ValueError(f"[{name}] CSV must contain columns theta1 and theta2. Found: {list(df.columns)[:20]}")

    # robust index
    # if no dates in CSV, choose defaults by episode
    if ep["name"].lower().startswith("taka"):
        start, end = "1931-01-01", "1936-12-31"
    elif ep["name"].lower().startswith("abe"):
        start, end = "2013-01-01", "2024-12-31"
    else:
        start, end = "2000-01-01", "2030-12-31"

    df, has_real_dt = _safe_datetime_index(df, ep.get("date_col", None), start, end)
    dt_days = _infer_dt_days(df.index)

    # compute IG
    ig_mode = ep["ig_mode"]
    ig, ig_meta = _compute_ig(df, ig_mode, ep.get("ig_params", {}))

    # compute STI proxy
    sti = _sti_proxy(df, ep["normal"], ROLL_WIN)

    # composite score
    V = ig / (EPS + sti)

    # labels + threshold
    y, event_idxs, lead_steps, dt_days = _build_labels(df.index, ep["event_dates"], ep["lead_days"])
    tau, n_baseline = _calibrate_threshold(V, y, event_idxs, lead_steps, TARGET_FPR)
    roc, pr, n_pos, n_neg = _eval_metrics(V, y)
    lead_mean_m, lead_list_m = _lead_at_threshold(V, df.index, event_idxs, lead_steps, tau, dt_days)

    fig_path = _plot_episode(name, df, ig, sti, V, event_idxs, lead_steps, tau)
    generated_figs.append(fig_path)

    # one row per episode (if you want per-event rows, you can expand)
    rows.append(dict(
        Episode=ep["name"],
        ROC_AUC=roc,
        PR_AUC=pr,
        Lead_months=lead_mean_m,
        n_pos=n_pos,
        n_neg=n_neg,
        tau=tau,
        baseline_n=n_baseline,
        dt_days=dt_days,
        IG_meta=str(ig_meta)
    ))

    print(f"[OK] {ep['name']}: ROC={roc:.4f} PR={pr:.4f} Lead@FPR={TARGET_FPR:.2f} -> {lead_mean_m:.2f} months")
    print(f"     tau={tau}  baseline_n={n_baseline}  pos={n_pos} neg={n_neg}  dt_days≈{dt_days:.4g}")
    print(f"     IG_meta={ig_meta}")
pri    nt(f"     wrote {fig_path}")

# save CSV summary
df_out = pd.DataFrame(rows)
df_out.to_csv(OUTDIR / "table_ews_eval_full.csv", index=False)

# write LaTeX table (Table 7)
def fmt(x):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "NA"
    return f"{x:.3f}"

tex_lines = []
tex_lines.append(r"\begin{tabular}{lccc}")
tex_lines.append(r"\hline")
tex_lines.append(r"Episode / event definition & ROC--AUC & PR--AUC & Lead@FPR=5\% (months) \\")
tex_lines.append(r"\hline")
for r in rows:
    tex_lines.append(f"{r['Episode']} & {fmt(r['ROC_AUC'])} & {fmt(r['PR_AUC'])} & {fmt(r['Lead_months'])} \\\\")
tex_lines.append(r"\hline")
tex_lines.append(r"\end{tabular}")

(OUTDIR / "table_ews_eval.tex").write_text("\n".join(tex_lines), encoding="utf-8")
print(f"[OK] wrote {OUTDIR/'table_ews_eval.tex'}")
print(f"[OK] wrote {OUTDIR/'table_ews_eval_full.csv'}")
print("[DONE] Generated figures:")
for p in generated_figs:
    print("  -", p)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import datetime as dt

OUT_PATH = Path("/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ---- ここが本体：df_theta を必ず作る（あなたの変数名に合わせる）----
# df_theta には最低限 theta1, theta2 列が必要
# 例：すでに df_theta があるなら、この行は不要
# df_theta = your_dataframe

assert "theta1" in df_theta.columns and "theta2" in df_theta.columns, df_theta.columns

# date列が無いなら付ける（Abenomicsなど想定：2013-01-01〜2024-12-31）
if not any(c.lower() in ["date","datetime","time","timestamp","t"] for c in df_theta.columns):
    start = "2013-01-01"
    end   = "2024-12-31"
    df_theta = df_theta.copy()
    df_theta.insert(0, "date", pd.date_range(start=start, end=end, periods=len(df_theta)))

# 安全化（NaN除去・型統一）
df_theta["theta1"] = pd.to_numeric(df_theta["theta1"], errors="coerce").fillna(method="ffill").fillna(0.0)
df_theta["theta2"] = pd.to_numeric(df_theta["theta2"], errors="coerce").fillna(method="ffill").fillna(0.0)

# 上書き前バックアップ（任意：事故防止）
if OUT_PATH.exists():
    ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    bk = OUT_PATH.with_name(f"theta_timeseries_backup_{ts}.csv")
    OUT_PATH.replace(bk)
    print("[BACKUP] moved old ->", bk)

# 保存
df_theta.to_csv(OUT_PATH, index=False)
print("[OK] wrote:", OUT_PATH)

# 確認
print(pd.read_csv(OUT_PATH).head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'df_theta' is not defined

In [4]:
from google.colab import files
files.download("theta_timeseries_run0001.zip")

FileNotFoundError: Cannot find file: theta_timeseries_run0001.zip

In [5]:
!zip -j theta_timeseries_run0001.zip "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"

  adding: theta_timeseries.csv (deflated 75%)


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv")

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!mkdir -p "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001"

In [8]:
!if [ -f "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv" ]; then \
  mv "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv" \
     "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries_backup_$(date +%Y%m%d_%H%M%S).csv" ; \
fi

In [9]:
!cp "/content/theta_timeseries 3.csv" \
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"

cp: cannot stat '/content/theta_timeseries 3.csv': No such file or directory


In [10]:
!ls -lh "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
!head -n 3 "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"

ls: cannot access '/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv': No such file or directory
head: cannot open '/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv' for reading: No such file or directory


In [12]:
BASE_THETA_CSV = "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"

In [13]:
!ls -lh /content | head

total 64K
drwx------ 5 root root 4.0K Feb 25 09:02 drive
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data
-rw-r--r-- 1 root root  56K Feb 25 09:54 theta_timeseries_run0001.zip


In [14]:
import pandas as pd, numpy as np

CSV = "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
df = pd.read_csv(CSV)
print("CSV:", CSV)
print("cols:", df.columns.tolist())
print("N:", len(df))
print("theta1: min/max/std =", df["theta1"].min(), df["theta1"].max(), df["theta1"].std())
print("theta2: min/max/std =", df["theta2"].min(), df["theta2"].max(), df["theta2"].std())

# date列っぽいもの
date_col = None
for c in df.columns:
    if c.lower() in ["date","datetime","time","timestamp","t"]:
        date_col = c; break
print("date_col:", date_col)
if date_col:
    d = pd.to_datetime(df[date_col], errors="coerce")
    print("date range:", d.min(), "->", d.max(), " valid=", d.notna().sum())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv'

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!ls -lah /content/drive/MyDrive | head

total 1.5G
-rw------- 1 root root 1.5M Aug 19  2025 2023_09_04 20_36 Office Lens~2.jpg
drwx------ 2 root root 4.0K Feb 13 11:53 BID_Bi2Te3_rs_sensitivity_20260213.ipynb
drwx------ 2 root root 4.0K Aug 19  2025 CGTAD
drwx------ 5 root root 4.0K Aug 17  2025 CGTAD2
drwx------ 2 root root 4.0K Aug 18  2025 CGTAD_backups
drwx------ 3 root root 4.0K Aug  3  2025 CliffordProject
drwx------ 2 root root 4.0K Aug  6  2025 CliffordTools
drwx------ 9 root root 4.0K Aug 21  2025 colab_backup
drwx------ 3 root root 4.0K Feb 11 16:05 Colab_Backup


In [17]:
from pathlib import Path
hits = sorted(Path("/content/drive/MyDrive").rglob("theta_timeseries.csv"),
              key=lambda p: p.stat().st_mtime, reverse=True)
print("hits =", len(hits))
for p in hits[:10]:
    print(p)
BASE_THETA_CSV = str(hits[0])
print("USE:", BASE_THETA_CSV)

hits = 3
/content/drive/MyDrive/CGTAD2/_colab_checkpoint_2026-02-11/DTT_EXP/runs/run_0001/theta_timeseries.csv
/content/drive/MyDrive/DTT_EXP/runs/run_0001/theta_timeseries.csv
/content/drive/MyDrive/Colab_Backup/20260212_work/DTT_EXP/runs/run_0001/theta_timeseries.csv
USE: /content/drive/MyDrive/CGTAD2/_colab_checkpoint_2026-02-11/DTT_EXP/runs/run_0001/theta_timeseries.csv


In [20]:
!mkdir -p "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001"
# ↓↓↓ 下のSRCは、直前の find 出力で見えたCSVに置き換える
SRC="/content/drive/MyDrive/CGTAD2/_colab_checkpoint_2026-02-11/DTT_EXP/runs/run_0001/theta_timeseries.csv"
DST="/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
!cp "$SRC" "$DST"
!ls -lh "$DST"
!head -n 3 "$DST"

-rw------- 1 root root 1.7K Feb 25 11:40 /content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
t_sec,theta1,theta2
0.0,1,0.97
1.0,2,0.9409


In [21]:
import pandas as pd, numpy as np

CSV = "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
df = pd.read_csv(CSV)

print("CSV:", CSV)
print("shape:", df.shape)          # ← N(行数)を見る
print("cols:", df.columns.tolist())
print(df.head(3))
print(df.tail(3))
print("theta1 std:", df["theta1"].std(), " theta2 std:", df["theta2"].std())

CSV: /content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
shape: (64, 3)
cols: ['t_sec', 'theta1', 'theta2']
   t_sec  theta1    theta2
0    0.0       1  0.970000
1    1.0       2  0.940900
2    2.0       3  0.912673
    t_sec  theta1    theta2
61   61.0      62  0.151303
62   62.0      63  0.146764
63   63.0      64  0.142361
theta1 std: 18.618986725025255  theta2 std: 0.23846885066006002


In [22]:
!ls -lt "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001" | head -n 30

total 587
-rw------- 1 root root   1708 Feb 25 11:40 theta_timeseries.csv
-rw------- 1 root root 228898 Feb 25 11:21 theta_timeseries_backup_20260225_112107.csv
-rw------- 1 root root 368892 Feb 10 09:39 obs_scalar.csv


In [23]:
from pathlib import Path
import pandas as pd

d = Path("/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001")
for p in sorted(d.glob("*.csv"), key=lambda x: x.stat().st_size, reverse=True)[:20]:
    try:
        n = sum(1 for _ in open(p, "r", encoding="utf-8", errors="ignore")) - 1
    except:
        n = None
    print(p.name, "size=", p.stat().st_size, "rows=", n)

obs_scalar.csv size= 368892 rows= 19999
theta_timeseries_backup_20260225_112107.csv size= 228898 rows= 19999
theta_timeseries.csv size= 1708 rows= 64


In [27]:
# 1) いまの小さい theta_timeseries.csv を退避（任意）
!mv "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv" \
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries_small_64rows.csv"

# 2) 19999行のバックアップを正規名に戻す（これが本命）
!cp "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries_backup_20260225_112107.csv" \
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"

# 3) 行数チェック
!python - << 'PY'
import pandas as pd
p="/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
df=pd.read_csv(p)
print("shape:", df.shape)
print(df.head(2))
print(df.tail(2))

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
shape: (19999, 3)
   t_sec  theta1  theta2
0    0.0       5       0
1    1.0       7       0
         t_sec  theta1  theta2
19997  19997.0       9       0
19998  19998.0       5       0


In [25]:
import pandas as pd

p = "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
df = pd.read_csv(p)
print("shape:", df.shape)
print(df.head(2))
print(df.tail(2))

shape: (19999, 3)
   t_sec  theta1  theta2
0    0.0       5       0
1    1.0       7       0
         t_sec  theta1  theta2
19997  19997.0       9       0
19998  19998.0       5       0


In [26]:
import pandas as pd, numpy as np
p="/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv"
df=pd.read_csv(p)
print("theta1 std:", df["theta1"].std(), "theta2 std:", df["theta2"].std())
print("theta2 unique approx:", pd.Series(df["theta2"]).round(6).nunique())

theta1 std: 2.8928069021149443 theta2 std: 0.0
theta2 unique approx: 1


In [30]:
# =========================
# EWS (Event-level Early Warning) — one-cell runnable
# Outputs: outputs_ews/FigEWS_*.pdf and outputs_ews/table_ews_eval*.tex
# =========================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# CONFIG (EDIT HERE)
# -------------------------
OUTDIR = Path("outputs_ews")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Your theta_timeseries.csv candidates (Drive path)
CAND_THETA_CSVS = [
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv",
]

CASE_NAME = "Abenomics"

EPS = 1e-6
TARGET_FPR = 0.05

# "event" index positions in the loaded dataframe (your log showed these)
EVENT_IDXS = [6202, 16247]

# warning window length BEFORE each event (in samples)
WARN_WIN = 365   # 365 samples; change if your sampling is not daily-like

# minimum separation between triggers (avoid noisy multiple triggers)
REFRACTORY = 30

# rolling window for IG proxy (in samples)
ROLL_WIN = 12

# degenerate detection tolerance
DEGEN_TOL = 1e-12


# -------------------------
# Helpers
# -------------------------
def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("No existing theta CSV found in CAND_THETA_CSVS.")

def safe_auc(y, s):
    y = np.asarray(y)
    s = np.asarray(s)

    m = np.isfinite(s)  # s側のNaN/infを落とす
    y = y[m]
    s = s[m]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y)
    s = np.asarray(s)

    m = np.isfinite(s)
    y = y[m]
    s = s[m]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def build_datetime_index(n, start="2013-01-01", end="2024-12-31"):
    # Synthetic (as in your log) to allow readable x-axis; keeps spacing uniform
    return pd.date_range(start=start, end=end, periods=n)

def rolling_std(x, w):
    return pd.Series(x).rolling(w, min_periods=max(3, w//3)).std().to_numpy()

def rolling_absdiff(x, w):
    # IG proxy: rolling std of first difference (captures agitation)
    dx = np.diff(x, prepend=x[0])
    return rolling_std(dx, w)

def calibrate_threshold(scores, y_true, target_fpr):
    neg = scores[y_true == 0]
    neg = neg[np.isfinite(neg)]
    if len(neg) == 0:
        return np.nan
    # threshold so that P(score > thr | negative) = target_fpr
    q = 1.0 - float(target_fpr)
    return float(np.quantile(neg, q))

def event_trigger_delay(scores, thr, event_idxs, warn_win, refractory):
    """
    For each event, look back warn_win samples and find the earliest trigger (score > thr).
    Return delays (event_idx - trigger_idx). If no trigger, delay=nan.
    """
    delays = []
    trig_idxs = []
    last_trig = -10**18
    for e in event_idxs:
        a = max(0, e - warn_win)
        b = e
        trig = np.nan
        for t in range(a, b):
            if t - last_trig < refractory:
                continue
            if np.isfinite(scores[t]) and np.isfinite(thr) and (scores[t] > thr):
                trig = t
                last_trig = t
                break
        if np.isnan(trig):
            delays.append(np.nan)
            trig_idxs.append(np.nan)
        else:
            delays.append(float(e - trig))
            trig_idxs.append(int(trig))
    return np.array(delays), trig_idxs


# -------------------------
# Load data
# -------------------------
theta_csv = first_existing(CAND_THETA_CSVS)
df = pd.read_csv(theta_csv)

# expected columns: t_sec, theta1, theta2 (as in your screenshot)
if "theta1" not in df.columns:
    raise ValueError(f"theta1 not found. columns={list(df.columns)}")
if "theta2" not in df.columns:
    raise ValueError(f"theta2 not found. columns={list(df.columns)}")

n = len(df)
df["Date"] = build_datetime_index(n)
df = df.set_index("Date")

theta1 = df["theta1"].to_numpy(dtype=float)
theta2 = df["theta2"].to_numpy(dtype=float)

std1 = float(np.nanstd(theta1))
std2 = float(np.nanstd(theta2))
uniq2 = int(pd.Series(theta2).round(6).nunique(dropna=True))

print(f"[CONFIG] Using BASE_THETA_CSV = {theta_csv}")
print(f"[COLS] {CASE_NAME}: theta1=theta1, theta2=theta2")
print(f"[STAT] theta1 std={std1:.6g}  theta2 std={std2:.6g}  theta2 unique≈{uniq2}")

# -------------------------
# Build STI proxy and IG proxy (robust fallback)
# -------------------------
if std2 <= DEGEN_TOL or uniq2 <= 1:
    print(f"[WARN] {CASE_NAME}: theta2 degenerate -> using theta1-only proxies.")
    STIproxy = np.maximum(np.abs(theta1), np.quantile(np.abs(theta1[np.abs(theta1)>0]), 0.05))
    IG = rolling_absdiff(theta1, ROLL_WIN)
    STI_source = "theta1 (fallback; theta2 degenerate)"
else:
    # example 2D proxy (you can replace with your Appendix-consistent definitions)
    STIproxy = np.sqrt(theta1**2 + theta2**2)
    IG = np.sqrt(rolling_absdiff(theta1, ROLL_WIN)**2 + rolling_absdiff(theta2, ROLL_WIN)**2)
    STI_source = "sqrt(theta1^2+theta2^2)"

# EWS score
V = IG / (EPS + STIproxy)
V = np.asarray(V, dtype=float)

# -------------------------
# Build event-level labels
# y_true[t]=1 if t is in any pre-event warning window; else 0.
# -------------------------
y_true = np.zeros(n, dtype=int)
for e in EVENT_IDXS:
    a = max(0, e - WARN_WIN)
    b = min(n, e)  # strictly before event
    y_true[a:b] = 1

pos = int(y_true.sum())
neg = int((y_true == 0).sum())
print(f"[SANITY] {CASE_NAME}: N={n} pos={pos} neg={neg} event_pos={EVENT_IDXS}")
print(f"[SANITY-STATS] {CASE_NAME}: STI source = {STI_source}")

# -------------------------
# Threshold calibration on negatives (baseline) for target FPR
# -------------------------
thr = calibrate_threshold(V, y_true, TARGET_FPR)
print(f"[THR] calibrated threshold @FPR={TARGET_FPR:.0%}: thr={thr:.6g}")

# -------------------------
# Metrics
# -------------------------
roc = safe_auc(y_true, V)
pr  = safe_ap(y_true, V)

delays, trig_idxs = event_trigger_delay(V, thr, EVENT_IDXS, WARN_WIN, REFRACTORY)
delay_mean = float(np.nanmean(delays)) if np.isfinite(delays).any() else np.nan

print(f"[METRIC] ROC-AUC={roc:.6g}  PR-AUC={pr:.6g}  Delay(mean samples)={delay_mean:.6g}")
print(f"[TRIG] trigger indices={trig_idxs}")

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, V, lw=1.0, label="EWS score V_t")
if np.isfinite(thr):
    ax.axhline(thr, ls="--", lw=1.0, label=f"thr (FPR={TARGET_FPR:.0%})")

# shade warning windows + mark events
for e in EVENT_IDXS:
    a = max(0, e - WARN_WIN)
    ax.axvspan(df.index[a], df.index[e], alpha=0.12)
    ax.axvline(df.index[e], lw=1.0)

ax.set_title(f"EWS — {CASE_NAME}  (STI={STI_source})")
ax.set_ylabel("V_t")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

figpath = OUTDIR / f"FigEWS_{CASE_NAME}.pdf"
fig.tight_layout()
fig.savefig(figpath)
plt.close(fig)
print(f"[OK] wrote {figpath}")

# -------------------------
# Write TeX table with numbers (always non-empty)
# -------------------------
row = {
    "Case": CASE_NAME,
    "STI_source": STI_source,
    "ROC-AUC": float(roc) if roc == roc else np.nan,
    "PR-AUC": float(pr) if pr == pr else np.nan,
    f"Delay@FPR={TARGET_FPR:.0%}": float(delay_mean) if delay_mean == delay_mean else np.nan,
}
tbl = pd.DataFrame([row])

texpath = OUTDIR / f"table_ews_eval_{CASE_NAME}.tex"
tbl.to_latex(texpath, index=False, float_format="%.4f")
print(f"[OK] wrote {texpath}")

# quick peek
print("\n--- table preview ---")
print(tbl)

[CONFIG] Using BASE_THETA_CSV = /content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
[COLS] Abenomics: theta1=theta1, theta2=theta2
[STAT] theta1 std=2.89273  theta2 std=0  theta2 unique≈1
[WARN] Abenomics: theta2 degenerate -> using theta1-only proxies.
[SANITY] Abenomics: N=19999 pos=730 neg=19269 event_pos=[6202, 16247]
[SANITY-STATS] Abenomics: STI source = theta1 (fallback; theta2 degenerate)
[THR] calibrated threshold @FPR=5%: thr=4.83222
[METRIC] ROC-AUC=0.509985  PR-AUC=0.0394756  Delay(mean samples)=316
[TRIG] trigger indices=[5934, 15883]
[OK] wrote outputs_ews/FigEWS_Abenomics.pdf
[OK] wrote outputs_ews/table_ews_eval_Abenomics.tex

--- table preview ---
        Case                            STI_source   ROC-AUC    PR-AUC  \
0  Abenomics  theta1 (fallback; theta2 degenerate)  0.509985  0.039476   

   Delay@FPR=5%  
0         316.0  


In [32]:
# =========================
# EWS — sweep + pick best (one cell)
# Writes:
#   outputs_ews/table_ews_sweep_Abenomics.tex
#   outputs_ews/table_ews_eval_Abenomics.tex
#   outputs_ews/FigEWS_Abenomics.pdf
# =========================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# CONFIG
# -------------------------
OUTDIR = Path("outputs_ews")
OUTDIR.mkdir(parents=True, exist_ok=True)

CAND_THETA_CSVS = [
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv",
]

CASE_NAME = "Abenomics"

EVENT_IDXS = [6202, 16247]   # your log
TARGET_FPR = 0.05
EPS = 1e-6

# Sweep ranges (軽め)
WARN_WINS = [90, 180, 365, 540, 730]         # samples
ROLL_WINS = [6, 12, 24, 48]                  # samples
FLOOR_QS  = [0.01, 0.05, 0.10]               # STI floor quantile (positive part)
TRANSFORMS = ["raw", "log1p", "clip99_log1p"] # score transforms

REFRACTORY = 30
DEGEN_TOL = 1e-12

# -------------------------
# Helpers
# -------------------------
def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("No existing theta CSV found in CAND_THETA_CSVS.")

def build_datetime_index(n, start="2013-01-01", end="2024-12-31"):
    return pd.date_range(start=start, end=end, periods=n)

def rolling_std(x, w):
    return pd.Series(x).rolling(w, min_periods=max(3, w//3)).std().to_numpy()

def rolling_mean(x, w):
    return pd.Series(x).rolling(w, min_periods=max(3, w//3)).mean().to_numpy()

def safe_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    m = np.isfinite(s)
    y = y[m]; s = s[m]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y); s = np.asarray(s)
    m = np.isfinite(s)
    y = y[m]; s = s[m]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def calibrate_threshold(scores, y_true, target_fpr):
    scores = np.asarray(scores, float)
    y_true = np.asarray(y_true, int)
    neg = scores[(y_true == 0) & np.isfinite(scores)]
    if len(neg) == 0:
        return np.nan
    return float(np.quantile(neg, 1.0 - float(target_fpr)))

def scan_triggers(scores, thr, refractory):
    scores = np.asarray(scores, float)
    idxs = []
    last = -10**18
    if not np.isfinite(thr):
        return idxs
    for t in range(len(scores)):
        if t - last < refractory:
            continue
        if np.isfinite(scores[t]) and scores[t] > thr:
            idxs.append(t)
            last = t
    return idxs

def event_delays(scores, thr, event_idxs, warn_win, refractory):
    scores = np.asarray(scores, float)
    delays = []
    trig_idxs = []
    last_trig = -10**18
    for e in event_idxs:
        a = max(0, e - warn_win)
        b = e
        trig = np.nan
        for t in range(a, b):
            if t - last_trig < refractory:
                continue
            if np.isfinite(thr) and np.isfinite(scores[t]) and (scores[t] > thr):
                trig = t
                last_trig = t
                break
        if np.isnan(trig):
            delays.append(np.nan); trig_idxs.append(np.nan)
        else:
            delays.append(float(e - trig)); trig_idxs.append(int(trig))
    return np.array(delays), trig_idxs

def make_labels(n, event_idxs, warn_win):
    y = np.zeros(n, dtype=int)
    for e in event_idxs:
        a = max(0, e - warn_win)
        b = min(n, e)
        y[a:b] = 1
    return y

def sti_floor(sti, q):
    sti = np.asarray(sti, float)
    pos = sti[np.isfinite(sti) & (sti > 0)]
    if len(pos) == 0:
        return 1.0
    return float(np.quantile(pos, q))

def transform_score(v, mode):
    v = np.asarray(v, float)
    if mode == "raw":
        return v
    if mode == "log1p":
        return np.log1p(np.maximum(v, 0.0))
    if mode == "clip99_log1p":
        vv = v.copy()
        fin = np.isfinite(vv)
        if fin.any():
            c = np.quantile(vv[fin], 0.99)
            vv[fin] = np.minimum(vv[fin], c)
        return np.log1p(np.maximum(vv, 0.0))
    raise ValueError(mode)

# IG/STI variants (theta2 degenerate前提の1D)
def compute_ig(theta1, roll, mode):
    x = np.asarray(theta1, float)
    dx = np.diff(x, prepend=x[0])
    if mode == "std_level":
        return rolling_std(x, roll)
    if mode == "std_diff":
        return rolling_std(dx, roll)
    if mode == "mean_absdiff":
        return rolling_mean(np.abs(dx), roll)
    if mode == "mean_level":
        return rolling_mean(np.abs(x), roll)
    raise ValueError(mode)

def compute_sti(theta1, roll, mode):
    x = np.asarray(theta1, float)
    dx = np.diff(x, prepend=x[0])
    if mode == "abs":
        return np.abs(x)
    if mode == "roll_mean_abs":
        return rolling_mean(np.abs(x), roll)
    if mode == "roll_std_level":
        return rolling_std(x, roll)
    if mode == "roll_mean_absdiff":
        return rolling_mean(np.abs(dx), roll)
    raise ValueError(mode)

IG_MODES  = ["std_level", "std_diff", "mean_absdiff"]
STI_MODES = ["abs", "roll_mean_abs", "roll_std_level"]

# -------------------------
# Load data
# -------------------------
theta_csv = first_existing(CAND_THETA_CSVS)
df = pd.read_csv(theta_csv)

for c in ["theta1", "theta2"]:
    if c not in df.columns:
        raise ValueError(f"{c} not found. columns={list(df.columns)}")

n = len(df)
df["Date"] = build_datetime_index(n)
df = df.set_index("Date")

theta1 = df["theta1"].to_numpy(dtype=float)
theta2 = df["theta2"].to_numpy(dtype=float)

std1 = float(np.nanstd(theta1))
std2 = float(np.nanstd(theta2))
uniq2 = int(pd.Series(theta2).round(6).nunique(dropna=True))

print(f"[CONFIG] Using BASE_THETA_CSV = {theta_csv}")
print(f"[STAT] theta1 std={std1:.6g}  theta2 std={std2:.6g}  theta2 unique≈{uniq2}")
if std2 <= DEGEN_TOL or uniq2 <= 1:
    print(f"[WARN] {CASE_NAME}: theta2 degenerate -> sweep uses theta1-only.")
else:
    print(f"[NOTE] theta2 is not degenerate; (this sweep is still theta1-only by design).")

# -------------------------
# Sweep
# -------------------------
rows = []
best = None

for warn_win in WARN_WINS:
    y_true = make_labels(n, EVENT_IDXS, warn_win)
    pos = int(y_true.sum()); neg = int((y_true==0).sum())
    base_rate = pos / n

    for roll in ROLL_WINS:
        for ig_mode in IG_MODES:
            IG = compute_ig(theta1, roll, ig_mode)

            for sti_mode in STI_MODES:
                STI = compute_sti(theta1, roll, sti_mode)

                for fq in FLOOR_QS:
                    floor = sti_floor(STI, fq)
                    denom = EPS + np.maximum(STI, floor)
                    V_raw = IG / denom

                    for tfm in TRANSFORMS:
                        V = transform_score(V_raw, tfm)

                        thr = calibrate_threshold(V, y_true, TARGET_FPR)
                        roc = safe_auc(y_true, V)
                        pr  = safe_ap(y_true, V)

                        delays, trig_idxs = event_delays(V, thr, EVENT_IDXS, warn_win, REFRACTORY)
                        delay_mean = float(np.nanmean(delays)) if np.isfinite(delays).any() else np.nan

                        # trigger stats
                        trig_all = scan_triggers(V, thr, REFRACTORY)
                        false_trigs = sum(1 for t in trig_all if y_true[t] == 0)
                        true_trigs  = sum(1 for t in trig_all if y_true[t] == 1)

                        # event hit rate: each event has at least one trigger in its pre-window?
                        hit = 0
                        for e in EVENT_IDXS:
                            a = max(0, e - warn_win); b = e
                            if any((t>=a and t<b) for t in trig_all):
                                hit += 1

                        pr_gain = (pr - base_rate) if np.isfinite(pr) else np.nan

                        row = dict(
                            Case=CASE_NAME,
                            warn_win=warn_win,
                            roll_win=roll,
                            ig=ig_mode,
                            sti=sti_mode,
                            floor_q=fq,
                            tfm=tfm,
                            pos=pos,
                            base_rate=base_rate,
                            ROC_AUC=roc,
                            PR_AUC=pr,
                            PR_gain=pr_gain,
                            Delay_mean=delay_mean,
                            Event_hits=hit,
                            Trig_true=true_trigs,
                            Trig_false=false_trigs,
                            thr=thr,
                        )
                        rows.append(row)

                        # Selection rule: maximize PR_gain, then ROC, then delay smaller, then fewer false triggers
                        key = (
                            row["PR_gain"] if np.isfinite(row["PR_gain"]) else -1e9,
                            row["ROC_AUC"] if np.isfinite(row["ROC_AUC"]) else -1e9,
                            -(row["Delay_mean"] if np.isfinite(row["Delay_mean"]) else 1e9),
                            -(row["Trig_false"]),
                        )
                        if (best is None) or (key > best[0]):
                            best = (key, row, V, y_true)

res = pd.DataFrame(rows)
res_sorted = res.sort_values(
    ["PR_gain","ROC_AUC","Delay_mean","Trig_false"],
    ascending=[False,False,True,True]
)

# ===== Override BEST selection (recommended for paper) =====
MAX_FALSE = 100          # 偽警報の上限（調整可）
FIX_WARN_WIN = 365       # 1年窓（調整可。Noneなら固定しない）

cand = res.copy()
cand = cand[cand["Event_hits"] == len(EVENT_IDXS)]  # 全イベント捕捉は必須

if FIX_WARN_WIN is not None:
    cand = cand[cand["warn_win"] == FIX_WARN_WIN]

cand = cand[cand["Trig_false"] <= MAX_FALSE]

# ここがポイント：ROC優先、同点なら PR_gain、次に Delay短い、偽警報少ない
cand = cand.sort_values(["ROC_AUC","PR_gain","Delay_mean","Trig_false"],
                        ascending=[False, False, True, True])

if len(cand) == 0:
    print("[WARN] constrained-best is empty. Relax MAX_FALSE or FIX_WARN_WIN.")
else:
    best_row = cand.iloc[0].to_dict()
    print("[CONSTRAINED BEST]", best_row)




# -------------------------
# Write sweep table
# -------------------------
sweep_path = OUTDIR / f"table_ews_sweep_{CASE_NAME}.tex"
# 上位20だけTeX化（長すぎ防止）
topk = res_sorted.head(20).copy()
# 見やすい列だけ
cols = ["warn_win","roll_win","ig","sti","floor_q","tfm","pos","base_rate",
        "ROC_AUC","PR_AUC","PR_gain","Delay_mean","Event_hits","Trig_false"]
topk = topk[cols]
topk.to_latex(sweep_path, index=False, float_format="%.4f")
print(f"[OK] wrote {sweep_path}")

# -------------------------
# Adopt best + write final table + plot
# -------------------------
_, best_row, best_V, best_y = best
print("[BEST]", best_row)

# Final 1-row table (paper-facing)
final_tbl = pd.DataFrame([{
    "Case": CASE_NAME,
    "STI_source": "theta1-only (sweep best; theta2 degenerate)",
    "warn_win": best_row["warn_win"],
    "roll_win": best_row["roll_win"],
    "ig": best_row["ig"],
    "sti": best_row["sti"],
    "floor_q": best_row["floor_q"],
    "tfm": best_row["tfm"],
    "ROC-AUC": best_row["ROC_AUC"],
    "PR-AUC": best_row["PR_AUC"],
    "PR-gain": best_row["PR_gain"],
    f"Delay@FPR={TARGET_FPR:.0%}": best_row["Delay_mean"],
    "Event_hits": best_row["Event_hits"],
    "False_trigs": best_row["Trig_false"],
}])

final_path = OUTDIR / f"table_ews_eval_{CASE_NAME}.tex"
final_tbl.to_latex(final_path, index=False, float_format="%.4f")
print(f"[OK] wrote {final_path}")

# Plot best
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, best_V, lw=1.0, label="EWS score (best)")
thr_best = best_row["thr"]
if np.isfinite(thr_best):
    ax.axhline(thr_best, ls="--", lw=1.0, label=f"thr (FPR={TARGET_FPR:.0%})")

for e in EVENT_IDXS:
    a = max(0, e - int(best_row["warn_win"]))
    ax.axvspan(df.index[a], df.index[e], alpha=0.12)
    ax.axvline(df.index[e], lw=1.0)

ax.set_title(f"EWS — {CASE_NAME} (best sweep; theta1-only)")
ax.set_ylabel("score")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()

figpath = OUTDIR / f"FigEWS_{CASE_NAME}.pdf"
fig.savefig(figpath)
plt.close(fig)
print(f"[OK] wrote {figpath}")

print("\n[TOP5]")
print(res_sorted.head(5)[["warn_win","roll_win","ig","sti","floor_q","tfm","ROC_AUC","PR_AUC","PR_gain","Delay_mean","Event_hits","Trig_false"]])

[CONFIG] Using BASE_THETA_CSV = /content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
[STAT] theta1 std=2.89273  theta2 std=0  theta2 unique≈1
[WARN] Abenomics: theta2 degenerate -> sweep uses theta1-only.
[CONSTRAINED BEST] {'Case': 'Abenomics', 'warn_win': 365, 'roll_win': 48, 'ig': 'std_diff', 'sti': 'roll_std_level', 'floor_q': 0.01, 'tfm': 'clip99_log1p', 'pos': 730, 'base_rate': 0.03650182509125456, 'ROC_AUC': 0.5740873271663174, 'PR_AUC': 0.04517642799654559, 'PR_gain': 0.008674602905291026, 'Delay_mean': 350.0, 'Event_hits': 2, 'Trig_true': 6, 'Trig_false': 94, 'thr': 0.9507898615779401}
[OK] wrote outputs_ews/table_ews_sweep_Abenomics.tex
[BEST] {'Case': 'Abenomics', 'warn_win': 730, 'roll_win': 12, 'ig': 'std_diff', 'sti': 'roll_mean_abs', 'floor_q': 0.1, 'tfm': 'log1p', 'pos': 1460, 'base_rate': 0.07300365018250912, 'ROC_AUC': np.float64(0.5081998524899345), 'PR_AUC': np.float64(0.08173514776118367), 'PR_gain': np.float64(0.008731497578674544), 'Delay_mean

In [33]:
from pathlib import Path
print((Path("outputs_ews")/"table_ews_eval_Abenomics.tex").read_text()[:400])

\begin{tabular}{llrrllrlrrrrrr}
\toprule
Case & STI_source & warn_win & roll_win & ig & sti & floor_q & tfm & ROC-AUC & PR-AUC & PR-gain & Delay@FPR=5% & Event_hits & False_trigs \\
\midrule
Abenomics & theta1-only (sweep best; theta2 degenerate) & 730 & 12 & std_diff & roll_mean_abs & 0.1000 & log1p & 0.5082 & 0.0817 & 0.0087 & 650.0000 & 2 & 209 \\
\bottomrule
\end{tabular}



In [34]:
# ===== POST: overwrite eval/fig using CONSTRAINED BEST =====
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = Path("outputs_ews")
CASE_NAME = "Abenomics"
TARGET_FPR = 0.05
REFRACTORY = 30
EPS = 1e-6

# constraints you intended
FIX_WARN_WIN = 365
MAX_FALSE = 100

# ---- sanity: required objects exist? ----
for name in ["res","df","theta1","EVENT_IDXS","compute_ig","compute_sti","sti_floor","transform_score",
             "make_labels","calibrate_threshold","safe_auc","safe_ap","event_delays","scan_triggers"]:
    if name not in globals():
        raise RuntimeError(f"[ERROR] '{name}' not found. Run the sweep cell first.")

cand = res.copy()
cand = cand[cand["Event_hits"] == len(EVENT_IDXS)]
cand = cand[cand["warn_win"] == FIX_WARN_WIN]
cand = cand[cand["Trig_false"] <= MAX_FALSE]
cand = cand.sort_values(["ROC_AUC","PR_gain","Delay_mean","Trig_false"],
                        ascending=[False, False, True, True])

if len(cand) == 0:
    raise RuntimeError("[ERROR] constrained-best empty. Relax MAX_FALSE or FIX_WARN_WIN.")

best_row = cand.iloc[0].to_dict()
print("[CONSTRAINED BEST USED]", best_row)

def score_from_row(theta1, roll, ig_mode, sti_mode, floor_q, tfm):
    IG = compute_ig(theta1, roll, ig_mode)
    STI = compute_sti(theta1, roll, sti_mode)
    floor = sti_floor(STI, floor_q)
    denom = EPS + np.maximum(STI, floor)
    V_raw = IG / denom
    return transform_score(V_raw, tfm)

V_best = score_from_row(theta1,
                        int(best_row["roll_win"]),
                        best_row["ig"],
                        best_row["sti"],
                        float(best_row["floor_q"]),
                        best_row["tfm"])
y_best = make_labels(len(df), EVENT_IDXS, int(best_row["warn_win"]))
thr_best = calibrate_threshold(V_best, y_best, TARGET_FPR)

roc_best = safe_auc(y_best, V_best)
pr_best  = safe_ap(y_best, V_best)

delays_best, trig_idxs_best = event_delays(V_best, thr_best, EVENT_IDXS, int(best_row["warn_win"]), REFRACTORY)
delay_mean_best = float(np.nanmean(delays_best)) if np.isfinite(delays_best).any() else np.nan

trig_all_best = scan_triggers(V_best, thr_best, REFRACTORY)
false_best = sum(1 for t in trig_all_best if y_best[t] == 0)

print("[FINAL BEST METRICS]",
      "ROC", roc_best, "PR", pr_best, "Delay", delay_mean_best,
      "False", false_best, "TrigIdx", trig_idxs_best)

# ---- overwrite eval table (paper-facing, settings included) ----
final_tbl = pd.DataFrame([{
    "Case": CASE_NAME,
    "STI_source": "theta1-only (theta2 degenerate; constrained-best)",
    "warn_win": int(best_row["warn_win"]),
    "roll_win": int(best_row["roll_win"]),
    "ig": best_row["ig"],
    "sti": best_row["sti"],
    "floor_q": float(best_row["floor_q"]),
    "tfm": best_row["tfm"],
    "ROC-AUC": roc_best,
    "PR-AUC": pr_best,
    "Delay@FPR=5%": delay_mean_best,
    "False_trigs": int(false_best),
    "Event_hits": int(best_row["Event_hits"]),
}])

texpath = OUTDIR / f"table_ews_eval_{CASE_NAME}.tex"
final_tbl.to_latex(texpath, index=False, float_format="%.4f")
print("[OK] overwrote", texpath)

# ---- overwrite figure ----
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(df.index, V_best, lw=1.0, label="EWS score (constrained-best)")
if np.isfinite(thr_best):
    ax.axhline(thr_best, ls="--", lw=1.0, label=f"thr (FPR={TARGET_FPR:.0%})")
for e in EVENT_IDXS:
    a = max(0, e - int(best_row["warn_win"]))
    ax.axvspan(df.index[a], df.index[e], alpha=0.12)
    ax.axvline(df.index[e], lw=1.0)
ax.set_title(f"EWS — {CASE_NAME} (constrained-best; theta1-only)")
ax.set_ylabel("score"); ax.grid(True, alpha=0.3); ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(OUTDIR / f"FigEWS_{CASE_NAME}.pdf")
plt.close(fig)
print("[OK] overwrote FigEWS")

[CONSTRAINED BEST USED] {'Case': 'Abenomics', 'warn_win': 365, 'roll_win': 48, 'ig': 'std_diff', 'sti': 'roll_std_level', 'floor_q': 0.01, 'tfm': 'clip99_log1p', 'pos': 730, 'base_rate': 0.03650182509125456, 'ROC_AUC': 0.5740873271663174, 'PR_AUC': 0.04517642799654559, 'PR_gain': 0.008674602905291026, 'Delay_mean': 350.0, 'Event_hits': 2, 'Trig_true': 6, 'Trig_false': 94, 'thr': 0.9507898615779401}
[FINAL BEST METRICS] ROC 0.5740873271663174 PR 0.04517642799654559 Delay 350.0 False 94 TrigIdx [5867, 15882]
[OK] overwrote outputs_ews/table_ews_eval_Abenomics.tex
[OK] overwrote FigEWS


In [35]:
from pathlib import Path
print((Path("outputs_ews")/"table_ews_eval_Abenomics.tex").read_text()[:400])

\begin{tabular}{llrrllrlrrrrr}
\toprule
Case & STI_source & warn_win & roll_win & ig & sti & floor_q & tfm & ROC-AUC & PR-AUC & Delay@FPR=5% & False_trigs & Event_hits \\
\midrule
Abenomics & theta1-only (theta2 degenerate; constrained-best) & 365 & 48 & std_diff & roll_std_level & 0.0100 & clip99_log1p & 0.5741 & 0.0452 & 350.0000 & 94 & 2 \\
\bottomrule
\end{tabular}



In [36]:
# =========================
# EWS (Event-level Early Warning) — ONE CELL COMPLETE (Colab-ready)
# - Sweep hyperparams
# - Pick constrained-best (paper-facing)
# - Write final TeX table + PDF figure (legend outside, no overlap)
# Outputs:
#   outputs_ews/FigEWS_Abenomics.pdf
#   outputs_ews/table_ews_eval_Abenomics.tex
#   outputs_ews/table_ews_sweep_Abenomics.tex
# =========================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# CONFIG (EDIT HERE)
# -------------------------
OUTDIR = Path("outputs_ews")
OUTDIR.mkdir(parents=True, exist_ok=True)

CASE_NAME = "Abenomics"

# Your theta_timeseries.csv candidates (Drive path)
CAND_THETA_CSVS = [
    "/content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv",
]

# Event indices (from your sanity log)
EVENT_IDXS = [6202, 16247]

# EWS calibration
TARGET_FPR = 0.05
EPS = 1e-6
REFRACTORY = 30

# Constrained-best selection (paper-facing)
FIX_WARN_WIN = 365     # 1-year pre-warning window (samples)
MAX_FALSE    = 100     # cap false triggers (adjust if needed)

# Sweep ranges (moderate size)
WARN_WINS = [90, 180, 365, 540, 730]
ROLL_WINS = [6, 12, 24, 48]
FLOOR_QS  = [0.01, 0.05, 0.10]     # STI floor quantile on positive support
TRANSFORMS = ["raw", "log1p", "clip99_log1p"]

# theta2 degenerate tolerance
DEGEN_TOL = 1e-12

# -------------------------
# Helpers
# -------------------------
def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("No existing theta CSV found in CAND_THETA_CSVS.")

def build_datetime_index(n, start="2013-01-01", end="2024-12-31"):
    # synthetic uniform index (for plotting readability)
    return pd.date_range(start=start, end=end, periods=n)

def rolling_std(x, w):
    return pd.Series(x).rolling(w, min_periods=max(3, w//3)).std().to_numpy()

def rolling_mean(x, w):
    return pd.Series(x).rolling(w, min_periods=max(3, w//3)).mean().to_numpy()

def safe_auc(y, s):
    y = np.asarray(y)
    s = np.asarray(s)
    m = np.isfinite(s)
    y = y[m]; s = s[m]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y)
    s = np.asarray(s)
    m = np.isfinite(s)
    y = y[m]; s = s[m]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def calibrate_threshold(scores, y_true, target_fpr):
    scores = np.asarray(scores, float)
    y_true = np.asarray(y_true, int)
    neg = scores[(y_true == 0) & np.isfinite(scores)]
    if len(neg) == 0:
        return np.nan
    return float(np.quantile(neg, 1.0 - float(target_fpr)))

def make_labels(n, event_idxs, warn_win):
    y = np.zeros(n, dtype=int)
    for e in event_idxs:
        a = max(0, e - warn_win)
        b = min(n, e)      # strictly before event
        y[a:b] = 1
    return y

def scan_triggers(scores, thr, refractory):
    scores = np.asarray(scores, float)
    if not np.isfinite(thr):
        return []
    idxs = []
    last = -10**18
    for t in range(len(scores)):
        if t - last < refractory:
            continue
        if np.isfinite(scores[t]) and scores[t] > thr:
            idxs.append(t)
            last = t
    return idxs

def event_delays(scores, thr, event_idxs, warn_win, refractory):
    scores = np.asarray(scores, float)
    delays = []
    trig_idxs = []
    last_trig = -10**18
    for e in event_idxs:
        a = max(0, e - warn_win)
        b = e
        trig = np.nan
        for t in range(a, b):
            if t - last_trig < refractory:
                continue
            if np.isfinite(thr) and np.isfinite(scores[t]) and scores[t] > thr:
                trig = t
                last_trig = t
                break
        if np.isnan(trig):
            delays.append(np.nan); trig_idxs.append(np.nan)
        else:
            delays.append(float(e - trig)); trig_idxs.append(int(trig))
    return np.array(delays), trig_idxs

def sti_floor(sti, q):
    sti = np.asarray(sti, float)
    pos = sti[np.isfinite(sti) & (sti > 0)]
    if len(pos) == 0:
        return 1.0
    return float(np.quantile(pos, q))

def transform_score(v, mode):
    v = np.asarray(v, float)
    if mode == "raw":
        return v
    if mode == "log1p":
        return np.log1p(np.maximum(v, 0.0))
    if mode == "clip99_log1p":
        vv = v.copy()
        fin = np.isfinite(vv)
        if fin.any():
            c = np.quantile(vv[fin], 0.99)
            vv[fin] = np.minimum(vv[fin], c)
        return np.log1p(np.maximum(vv, 0.0))
    raise ValueError(mode)

# IG/STI modes (theta1-only fallback)
IG_MODES  = ["std_level", "std_diff", "mean_absdiff"]
STI_MODES = ["abs", "roll_mean_abs", "roll_std_level"]

def compute_ig(theta1, roll, mode):
    x = np.asarray(theta1, float)
    dx = np.diff(x, prepend=x[0])
    if mode == "std_level":
        return rolling_std(x, roll)
    if mode == "std_diff":
        return rolling_std(dx, roll)
    if mode == "mean_absdiff":
        return rolling_mean(np.abs(dx), roll)
    raise ValueError(mode)

def compute_sti(theta1, roll, mode):
    x = np.asarray(theta1, float)
    if mode == "abs":
        return np.abs(x)
    if mode == "roll_mean_abs":
        return rolling_mean(np.abs(x), roll)
    if mode == "roll_std_level":
        return rolling_std(x, roll)
    raise ValueError(mode)

def score_from_params(theta1, roll, ig_mode, sti_mode, floor_q, tfm):
    IG  = compute_ig(theta1, roll, ig_mode)
    STI = compute_sti(theta1, roll, sti_mode)
    floor = sti_floor(STI, floor_q)
    denom = EPS + np.maximum(STI, floor)
    V_raw = IG / denom
    return transform_score(V_raw, tfm)

# -------------------------
# Load data
# -------------------------
theta_csv = first_existing(CAND_THETA_CSVS)
df0 = pd.read_csv(theta_csv)

for c in ["theta1", "theta2"]:
    if c not in df0.columns:
        raise ValueError(f"{c} not found. columns={list(df0.columns)}")

n = len(df0)
df0["Date"] = build_datetime_index(n)
df = df0.set_index("Date")

theta1 = df["theta1"].to_numpy(dtype=float)
theta2 = df["theta2"].to_numpy(dtype=float)

std1 = float(np.nanstd(theta1))
std2 = float(np.nanstd(theta2))
uniq2 = int(pd.Series(theta2).round(6).nunique(dropna=True))

print(f"[CONFIG] Using BASE_THETA_CSV = {theta_csv}")
print(f"[STAT] theta1 std={std1:.6g}  theta2 std={std2:.6g}  theta2 unique≈{uniq2}")
if std2 <= DEGEN_TOL or uniq2 <= 1:
    print(f"[WARN] {CASE_NAME}: theta2 degenerate -> sweep uses theta1-only.")
else:
    print(f"[NOTE] {CASE_NAME}: theta2 not degenerate (still using theta1-only sweep by design).")

# -------------------------
# Sweep
# -------------------------
rows = []
for warn_win in WARN_WINS:
    y_true = make_labels(n, EVENT_IDXS, warn_win)
    pos = int(y_true.sum())
    base_rate = pos / n

    for roll in ROLL_WINS:
        for ig_mode in IG_MODES:
            for sti_mode in STI_MODES:
                for fq in FLOOR_QS:
                    for tfm in TRANSFORMS:
                        V = score_from_params(theta1, roll, ig_mode, sti_mode, fq, tfm)
                        thr = calibrate_threshold(V, y_true, TARGET_FPR)

                        roc = safe_auc(y_true, V)
                        pr  = safe_ap(y_true, V)

                        delays, trig_idxs = event_delays(V, thr, EVENT_IDXS, warn_win, REFRACTORY)
                        delay_mean = float(np.nanmean(delays)) if np.isfinite(delays).any() else np.nan

                        trig_all = scan_triggers(V, thr, REFRACTORY)
                        trig_false = sum(1 for t in trig_all if y_true[t] == 0)

                        # event hit count (each event has at least one trigger in its pre-window)
                        hit = 0
                        for e in EVENT_IDXS:
                            a = max(0, e - warn_win); b = e
                            if any((t >= a and t < b) for t in trig_all):
                                hit += 1

                        pr_gain = (pr - base_rate) if np.isfinite(pr) else np.nan

                        rows.append(dict(
                            Case=CASE_NAME,
                            warn_win=warn_win,
                            roll_win=roll,
                            ig=ig_mode,
                            sti=sti_mode,
                            floor_q=fq,
                            tfm=tfm,
                            pos=pos,
                            base_rate=base_rate,
                            ROC_AUC=roc,
                            PR_AUC=pr,
                            PR_gain=pr_gain,
                            Delay_mean=delay_mean,
                            Event_hits=hit,
                            Trig_false=trig_false,
                            thr=thr,
                        ))

res = pd.DataFrame(rows)

# Sort for reporting (top by PR_gain then ROC then delay)
res_sorted = res.sort_values(
    ["PR_gain","ROC_AUC","Delay_mean","Trig_false"],
    ascending=[False, False, True, True]
)

# Write sweep table (top 20)
sweep_path = OUTDIR / f"table_ews_sweep_{CASE_NAME}.tex"
topk = res_sorted.head(20).copy()
cols = ["warn_win","roll_win","ig","sti","floor_q","tfm","pos","base_rate",
        "ROC_AUC","PR_AUC","PR_gain","Delay_mean","Event_hits","Trig_false"]
topk = topk[cols]
topk.to_latex(sweep_path, index=False, float_format="%.4f")
print(f"[OK] wrote {sweep_path}")

# -------------------------
# Pick CONSTRAINED BEST (paper-facing)
# If empty, relax constraints progressively.
# -------------------------
def pick_best(df_cand):
    # must hit all events
    df_cand = df_cand[df_cand["Event_hits"] == len(EVENT_IDXS)]
    if len(df_cand) == 0:
        return None
    # objective: ROC, then PR_gain, then shorter delay, then fewer false
    df_cand = df_cand.sort_values(
        ["ROC_AUC","PR_gain","Delay_mean","Trig_false"],
        ascending=[False, False, True, True]
    )
    return df_cand.iloc[0].to_dict()

cand = res.copy()
cand1 = cand[(cand["warn_win"] == FIX_WARN_WIN) & (cand["Trig_false"] <= MAX_FALSE)]
best_row = pick_best(cand1)

if best_row is None:
    print("[WARN] constrained-best empty: relaxing MAX_FALSE only")
    cand2 = cand[cand["warn_win"] == FIX_WARN_WIN]
    best_row = pick_best(cand2)

if best_row is None:
    print("[WARN] still empty: relaxing FIX_WARN_WIN too (global best)")
    best_row = pick_best(cand)

if best_row is None:
    raise RuntimeError("[ERROR] no candidate hits all events under any setting.")

print("[FINAL PICK]", best_row)

# -------------------------
# Recompute final score + metrics using best_row
# -------------------------
V_best = score_from_params(
    theta1,
    int(best_row["roll_win"]),
    best_row["ig"],
    best_row["sti"],
    float(best_row["floor_q"]),
    best_row["tfm"],
)

y_best = make_labels(n, EVENT_IDXS, int(best_row["warn_win"]))
thr_best = calibrate_threshold(V_best, y_best, TARGET_FPR)

roc_best = safe_auc(y_best, V_best)
pr_best  = safe_ap(y_best, V_best)

delays_best, trig_idxs_best = event_delays(V_best, thr_best, EVENT_IDXS, int(best_row["warn_win"]), REFRACTORY)
delay_mean_best = float(np.nanmean(delays_best)) if np.isfinite(delays_best).any() else np.nan

trig_all_best = scan_triggers(V_best, thr_best, REFRACTORY)
false_best = sum(1 for t in trig_all_best if y_best[t] == 0)

print("[FINAL METRICS]",
      f"ROC={roc_best:.6g}", f"PR={pr_best:.6g}",
      f"Delay={delay_mean_best:.6g}", f"False={false_best}",
      f"TrigIdx={trig_idxs_best}")

# -------------------------
# Write final TeX table (always non-empty)
# -------------------------
final_tbl = pd.DataFrame([{
    "Case": CASE_NAME,
    "STI_source": "theta1-only (theta2 degenerate; constrained-best)",
    "warn_win": int(best_row["warn_win"]),
    "roll_win": int(best_row["roll_win"]),
    "ig": best_row["ig"],
    "sti": best_row["sti"],
    "floor_q": float(best_row["floor_q"]),
    "tfm": best_row["tfm"],
    "ROC-AUC": float(roc_best) if roc_best == roc_best else np.nan,
    "PR-AUC": float(pr_best) if pr_best == pr_best else np.nan,
    f"Delay@FPR={TARGET_FPR:.0%}": float(delay_mean_best) if delay_mean_best == delay_mean_best else np.nan,
    "False_trigs": int(false_best),
    "Event_hits": int(best_row["Event_hits"]),
}])

eval_path = OUTDIR / f"table_ews_eval_{CASE_NAME}.tex"
final_tbl.to_latex(eval_path, index=False, float_format="%.4f")
print(f"[OK] wrote {eval_path}")

# -------------------------
# Plot final figure (legend outside; no overlap)
# -------------------------
fig, ax = plt.subplots(figsize=(12.5, 4.2))
ax.plot(df.index, V_best, lw=1.0, label="EWS score (final)")

if np.isfinite(thr_best):
    ax.axhline(thr_best, ls="--", lw=1.0, label=f"thr (FPR={TARGET_FPR:.0%})")

# warning windows + event lines
for e in EVENT_IDXS:
    a = max(0, e - int(best_row["warn_win"]))
    ax.axvspan(df.index[a], df.index[e], alpha=0.12)
    ax.axvline(df.index[e], lw=1.0)

# mark trigger points (important check points)
for t in trig_idxs_best:
    if isinstance(t, (int, np.integer)) and (0 <= t < n):
        ax.plot(df.index[t], V_best[t], marker="o", ms=4)

ax.set_title(f"EWS — {CASE_NAME} (final; theta1-only)")
ax.set_ylabel("score")
ax.grid(True, alpha=0.3)

# Legend outside to avoid overlap
leg = ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, frameon=True)
leg.get_frame().set_alpha(0.95)
leg.get_frame().set_facecolor("white")

fig.tight_layout()

fig_path = OUTDIR / f"FigEWS_{CASE_NAME}.pdf"
fig.savefig(fig_path, bbox_inches="tight")
plt.close(fig)
print(f"[OK] wrote {fig_path}")

# -------------------------
# Final: show outputs
# -------------------------
print("\n[FILES]", sorted([p.name for p in OUTDIR.glob("*")]))
print("\n[FINAL TABLE PREVIEW]")
print(final_tbl)

[CONFIG] Using BASE_THETA_CSV = /content/drive/MyDrive/DTT_EXP_backup/runs/run_0001/theta_timeseries.csv
[STAT] theta1 std=2.89273  theta2 std=0  theta2 unique≈1
[WARN] Abenomics: theta2 degenerate -> sweep uses theta1-only.
[OK] wrote outputs_ews/table_ews_sweep_Abenomics.tex
[FINAL PICK] {'Case': 'Abenomics', 'warn_win': 365, 'roll_win': 48, 'ig': 'std_diff', 'sti': 'roll_std_level', 'floor_q': 0.01, 'tfm': 'clip99_log1p', 'pos': 730, 'base_rate': 0.03650182509125456, 'ROC_AUC': 0.5740873271663174, 'PR_AUC': 0.04517642799654559, 'PR_gain': 0.008674602905291026, 'Delay_mean': 350.0, 'Event_hits': 2, 'Trig_false': 94, 'thr': 0.9507898615779401}
[FINAL METRICS] ROC=0.574087 PR=0.0451764 Delay=350 False=94 TrigIdx=[5867, 15882]
[OK] wrote outputs_ews/table_ews_eval_Abenomics.tex
[OK] wrote outputs_ews/FigEWS_Abenomics.pdf

[FILES] ['FigEWS_Abenomics.pdf', 'table_ews_eval_Abenomics.tex', 'table_ews_sweep_Abenomics.tex']

[FINAL TABLE PREVIEW]
        Case                                  